# Tutorial 4: Train NicheTrans* on SMA data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_SMA import SMA

from utils.utils import *
from utils.utils_training_SMA import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_SMA.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.2, dropout_rate=0.1, n_source=3000, n_target=50, img_size=256, workers=4, path_img='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used/patches', rna_path='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used', msi_path='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used', max_epoch=40, stepsize=20, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = SMA(path_img=args.path_img, rna_path=args.rna_path, msi_path=args.msi_path, n_top_genes=args.n_source, n_top_targets=args.n_target)
trainloader, testloader = sma_dataloader(args, dataset)

# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
source_dimension, target_dimension = dataset.rna_length, dataset.msi_length
model = NicheTrans_img(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate)
if torch.cuda.is_available():
    model = nn.DataParallel(model).cuda()
else:
    model = model.to(device)
print(f"Using device: {device}")

------Calculating spatial graph...


The graph contains 12134 edges, 3120 cells.
3.8891 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 24190 edges, 3120 cells.
7.7532 neighbors per cell on average.


------Calculating spatial graph...


...

------Calculating spatial graph...


The graph contains 10360 edges, 2675 cells.
3.8729 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 20628 edges, 2675 cells.
7.7114 neighbors per cell on average.


=> SMA loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  Without filtering  6038 spots from     2 slides 
  test     |  Without filtering  2675 spots from     1 slides
  train    |  After filting  6005 spots from     2 slides 
  test     |  After filting  2655 spots from     1 slides
  ------------------------------
Using device: cpu


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, use_img=True, device=device)
    if args.stepsize > 0: scheduler.step()
    ################
    
pearson = test(model, testloader, use_img=True, device=device) 
torch.save(model.state_dict(), 'NicheTrans_*_SMA_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/40


Batch 187/187	 Loss 64.906540 (92.716232)
==> Epoch 2/40


Batch 187/187	 Loss 26.011620 (42.906986)
==> Epoch 3/40


Batch 187/187	 Loss 9.283804 (17.764969)
==> Epoch 4/40


Batch 187/187	 Loss 6.920265 (10.678314)
==> Epoch 5/40


...

Batch 187/187	 Loss 4.428214 (4.277758)
==> Epoch 38/40


Batch 187/187	 Loss 4.977547 (4.322138)
==> Epoch 39/40


Batch 187/187	 Loss 4.532236 (4.276224)
==> Epoch 40/40


Batch 187/187	 Loss 4.004037 (4.292241)


Testing Set: pearson correlation 0.3677 + 0.1168; spearman correlation 0.4222 + 0.0802; rmse 2.2779 + 1.5961
Finished. Total elapsed time (h:m:s): 4:04:56
